# SwitchLingua — per-token language labels

This notebook labels every token of `Shelton1013/SwitchLingua_text` (234,172 rows) with its language:

| Rows | Tagger |
|---|---|
| Cantonese, Chinese, Japanese, Korean, Russian, Arabic, and Devanagari Hindi | Unicode script (exact) |
| Latin-script first language with English as L2, including romanized Hindi | AnE-LID (XLM-R-large) |
| German → French | wordfreq frequency comparison |
| Named entities, in every row | AnE-NER → a separate `entities` flag. Names keep a **language** label, and count as a switch only with visible evidence. A Latin-script name inside a non-Latin row is English. In Latin-script rows, the name's words decide when they're clearly one language's ("Severance Season 2" → English); otherwise it takes its clause's language ("Leo Messi" inside Spanish → Spanish). In non-Latin rows, Latin-script names get AnE-LID's label instead of "English by script". |

## How to run
1. **Settings:** set **Accelerator** to *GPU T4 x2* and turn **Internet** on.
2. **Token:** add a Kaggle secret named `HF_TOKEN` (*Add-ons → Secrets*) holding a Hugging Face token with **write** access, and accept the dataset terms at huggingface.co/datasets/Shelton1013/SwitchLingua_text. The token reads the corpus and uploads the results.
3. **Run it:** click **Save Version → Save & Run All (Commit)**. It runs with the browser closed.
4. **Watch progress** on Hugging Face at `huggingface.co/datasets/<you>/switchlingua-tagged-v2`, which is **private**. A pair's files appear there as soon as it finishes.
5. **If the run dies:** just commit again. The notebook checks the HF repo and skips every part already uploaded, so at most one part (≤ 20k rows) is redone.
6. **Smoke test first (optional):** set `LIMIT = 50` and `HF_REPO = "<you>/switchlingua-tagged-test"` so the test doesn't mix with the real output.

By default, name positions are read from the v1 labels (`ENTITY_SOURCE`), whose tokens are identical, so AnE-NER doesn't have to run again. Set `ENTITY_SOURCE = None` to recompute them.

## What ends up on Hugging Face
- **`switchlingua_tagged_rows.csv`**, uploaded last, once every pair is done. It has one line per row:
  - `tokens` and `labels` hold aligned, space-joined values, so `zip(tokens.split(" "), labels.split(" "))` pairs each token with its label.
  - The label codes are `L1`, `L2` and `O` (punctuation or digits). Names have a language code like every other word, resolved with the default (script_words) rule.
  - `entities` is an aligned `0`/`1` string, where `1` means the token is part of a named entity.
  - `l2_runs` holds the length in words of each second-language run, i.e. how long each switch lasts, under the default rule. Punctuation is skipped.
  - `l2_runs_names_transparent` skips names entirely, and `l2_runs_names_script` makes every Latin-script name English. These are the two ablations.
  - `split` is the row-level train/val/test assignment (`row_id % 10`).
- **`tagged.parquet`:** the same data in the local pipeline's format. Put it in `data/` for `baseline_anticipation.py`, `train_anticipation.py` and the rest.
- **`rows/<L1>_<L2>/part-NNN.csv` and `parquet/<L1>_<L2>/part-NNN.parquet`:** the per-pair pieces, uploaded as each finishes.

Download everything with `huggingface-cli download <you>/switchlingua-tagged --repo-type dataset --local-dir data/hf`.

*Generated by `scripts/build_kaggle_notebook.py`; do not edit by hand.*

## Config

In [ ]:
LIMIT = None          # e.g. 50 -> first 50 rows of each language pair, for a smoke test
BATCH = 64            # rows per GPU batch; lower to 32 if CUDA runs out of memory
HF_REPO = None        # None -> "<your-username>/switchlingua-tagged-v2" (private)
ENTITY_SOURCE = "cristae/switchlingua-tagged"   # v1 labels: name positions = its NE tokens.
                                                # None -> run AnE-NER again (slow)
WINDOW = 16           # context tokens either side of a Latin name in script-tagged rows
PART_ROWS = 20_000    # rows per upload; big pairs (Cantonese: 119k) go up in parts
UPLOAD = True         # False -> write to ./hf_mirror instead (local testing)

## Setup

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "regex", "wordfreq", "pyarrow"],
               check=True)
import torch
print("torch", torch.__version__, "| CUDA devices:", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if not torch.cuda.is_available():
    print("WARNING: no GPU - enable Settings > Accelerator > GPU T4 x2, or this takes ~10 h.")

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("kaggle_out")
os.makedirs(os.path.join(WORK, "src"), exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working dir:", WORK)

## Pipeline source
The live `src/` modules, embedded verbatim.

In [ ]:
%%writefile src/__init__.py
# package marker

In [ ]:
%%writefile src/routing.py
"""Language-pair -> tagging-strategy lookup.

Two regimes, confirmed empirically against the cached SwitchLingua CSVs:

* ``unicode`` - first_language uses a script disjoint from Latin, so a token's
  script *is* its language. Exact, no model. ~68.7% of rows.
* ``model``   - first_language is Latin-script (or romanized), so only
  vocabulary separates it from the second language. Needs constrained LID.

Note the plan's routing table listed Arabic's pair as Arabic<->French; the data
says ``Arabic_eng.csv`` is Arabic<->English, and French appears as a *second*
language only in ``German_French.csv``. Nothing here hardcodes the second
language - callers pass the row's own ``second_language``.
"""

SCRIPT_DISJOINT_CONFIRMED = {
    "Cantonese", "Chinese", "Japanese", "Korean", "Russian", "Arabic",
}

# Resolved from the plan's `unicode_pending_check` gate: 54.7% of Hindi rows are
# romanized Hinglish ("Woh kal Prime Minister Carney se milne wali hain"), so
# Devanagari-vs-Latin is not a reliable split for this language. Routed to model.
NEEDS_MODEL = {
    "German", "Philippines", "Vietnamese", "Malay", "Spanish", "French",
    "Italian", "Hindi",
}

# fastText lid.176 label codes.
LANG_TO_CODE = {
    "German": "de", "Philippines": "tl", "Vietnamese": "vi", "Malay": "ms",
    "Spanish": "es", "French": "fr", "Italian": "it", "English": "en",
    "Hindi": "hi", "Russian": "ru", "Arabic": "ar", "Chinese": "zh",
    "Cantonese": "zh", "Japanese": "ja", "Korean": "ko",
}

# Script expected for each unicode-routed first_language. A token in one of
# these scripts is the first language; a Latin token is the second language.
_EXPECTED_SCRIPT = {
    "Cantonese": {"Han"},
    "Chinese": {"Han"},
    "Japanese": {"Han", "Hiragana", "Katakana"},
    "Korean": {"Hangul"},
    "Russian": {"Cyrillic"},
    "Arabic": {"Arabic"},
    "Hindi": {"Devanagari"},
}

# fastText's `hi` model is trained on Devanagari. Romanized Hindi tokens are
# out of distribution for every label, so rows here are flagged low-confidence
# rather than silently trusted.
ROMANIZATION_RISK = {"Hindi"}


# --- tier 3: context-aware LinCE BERT token classifiers ---------------------
#
# Measured on hand-labelled tokens from the real rows (NE excluded), fastText
# vs. the spaeng model used cross-lingually:
#
#     language      fastText   spaeng    -> routed to
#     German           100%      80%        fasttext
#     French            90%      68%        fasttext
#     Vietnamese        90%      88%        fasttext   (tie; diacritics carry it)
#     Italian           96%     100%        spaeng
#     Spanish           94%     100%        spaeng
#     Philippines       86%      94%        spaeng
#     Malay             75%      88%        spaeng
#     Hindi          broken       -         hineng
#
# spaeng transfers as a generic Latin-L1-vs-English discriminator to Romance and
# Austronesian languages but *regresses* Germanic/French, so routing is per
# language and empirical, not by family. hineng does not transfer at all (17%
# Malay / 64% Tagalog) - it is Hindi-only.
#
# Sample sizes are small (n=26-48 per language); treat Vietnamese and Italian,
# where the gap is inside a couple of tokens, as ties rather than firm results.
MODEL_REPO = {
    "spaeng": "sagorsarker/codeswitch-spaeng-lid-lince",
    "hineng": "sagorsarker/codeswitch-hineng-lid-lince",
}
MODEL_L1_LABEL = {"spaeng": "spa", "hineng": "hin"}
MODEL_FOR = {
    "Hindi": "hineng",
    "Spanish": "spaeng", "Italian": "spaeng",
    "Malay": "spaeng", "Philippines": "spaeng",
}


# --- tier 4: dictionary frequency comparison (measured best) ----------------
#
# Measured on data/gold_eval.json (324 hand-labelled tokens) and confirmed on
# 767 rows against the dataset's own cs_ratio_result:
#
#   language      dict-freq   best model      ratio err (dict vs prod)
#   Malay            100%      88% mBERT        0.103 vs 0.183
#   Philippines      100%      94% mBERT        0.147 vs 0.222
#   French           100%      90% fastText     0.121 vs 0.153
#   German           100%     100% fastText     0.133 vs 0.151
#   Vietnamese       100%      90% fastText     0.084 vs 0.093
#   Spanish          100%     100% mBERT        0.105 vs 0.104
#   Italian          100%     100% mBERT        0.132 vs 0.124
#   Hindi (roman)     66%     100% mBERT        - model required
#
# dict-freq wins or ties on every Latin-script pair and needs no model at all.
# It also handles a non-English second language (the German<->French rows),
# which neither LinCE model can, so fastText and mBERT-spaeng both retire.
#
# Romanized Hindi is the sole exception: wordfreq's `hi` list is Devanagari and
# romanized Hinglish has no standard orthography, so no word list can cover it.
# Four independent techniques agree the context-aware models are required there
# (fastText 56%, dict-freq 66%, mBERT 100%, XLM-R 100%, CANINE 98%).
USES_DICT = {
    "Spanish", "Italian", "Malay", "Philippines",
    "German", "French", "Vietnamese",
}




# --- tier 5: AnE-LID (current production for every "<Latin L1> vs English") --
#
# The gold set above saturated (every tier at ~99-100% on 26-59 self-labelled
# tokens), so the decision was made on human-annotated public data instead
# (scripts/eval_public.py, L1/EN tokens only):
#
#   set                      n     dict-freq   mBERT    AnE-LID
#   LinCE spaeng dev      31,667     94.1%     98.8%     99.4%
#   LinCE hineng dev      12,303     80.5%     97.4%     98.8%
#   Denglisch (de-en)     59,485     96.2%     53.0%     99.9%  (likely seen in training)
#
# dict-freq's "100%" on the gold set was ~94-96% on human labels. AnE-LID is
# multi-pair trained, so it also covers romanized Hindi and replaces mBERT
# hineng there. It only answers "English or not", so German->French keeps
# dict-freq. Where AnE and dict-freq agree, tokens are 99.3-100% correct.
ANE_L2 = "English"


def ane_to_language(label: str, first_language: str, second_language: str = "English") -> str:
    """Map a raw AnE-LID label to the row's language. ``Mixed`` (an English stem
    in L1 morphology, "rewatchen") goes to L1, its grammatical host."""
    if label == "English":
        return second_language
    if label in ("notEnglish", "Mixed"):
        return first_language
    return "other"


def route(first_language: str, second_language: str = "English") -> str:
    """Return ``"unicode"``, ``"ane"`` or ``"dict"`` for a row.

    Hindi routes per *row* rather than per language - see
    ``dispatch._hindi_is_devanagari``. Devanagari rows are script-disjoint from
    English and go to the exact tagger; only romanized rows need the model.
    ``"bert"`` (mBERT hineng) is retired but still dispatchable for comparison.
    """
    if first_language in SCRIPT_DISJOINT_CONFIRMED:
        return "unicode"
    if second_language != ANE_L2:
        return "dict"
    return "ane"


def lang_code(language: str) -> str | None:
    """Map a dataset language name to its fastText lid.176 label code."""
    return LANG_TO_CODE.get(language)


def expected_script(language: str) -> set[str]:
    """Unicode script name(s) that indicate ``language`` in a mixed token stream."""
    return _EXPECTED_SCRIPT.get(language, {"Latin"})


# --- V1 quality tiers (measured, not assumed) -------------------------------
#
# The plan's escalation rule needs a per-pair accuracy number. Manual labelling
# was not required to get one: the six script-disjoint languages are tagged
# *exactly*, so their disagreement with the dataset's own `computed_ratio`
# (-0.134 .. +0.117, mean |diff| ~0.075) measures the noise in that reference
# signal rather than any tagger error. That band is the yardstick - a
# model-routed pair sitting inside it is behaving like an exact tagger; one far
# outside it is genuinely broken.
#
# Measured on a 150-row-per-language stratified sample (see notebook cell):
#
#   PASS  (inside the exact-tagger band)   Vietnamese .056  Spanish .086
#                                          German .118  French .128  Italian .129
#   FAIL  (2-5x outside it)                Malay .277  Hindi .369  Philippines .406
#
# This inverts the plan's expectation, which predicted German/Malay/Tagalog
# would pass easily and the Romance pairs would fail on shared function words.
# The Romance pairs are fine; lid.176 simply has far less Malay/Tagalog training
# data, and romanized Hindi is not in its label space at all.
V1_PASSES = {"Vietnamese", "Spanish", "German", "French", "Italian"}
V1_FAILS_NEEDS_V2 = {"Malay", "Philippines", "Hindi"}


# Measured token accuracy after routing each language to its best tagger.
# `low` now means "still under the plan's 90% bar with the best option tried".
# Accuracy of the tagger each language is now routed to, on the gold set.
# Every Latin pair scored 100% under dict-freq - but on only 26-50 tokens each,
# self-labelled. `provisional` marks that: good enough to ship, not yet a
# number to publish. The review tool exists to replace these.
_MEASURED_ACCURACY = {
    "Spanish": 1.00, "Italian": 1.00, "Malay": 1.00, "Philippines": 1.00,
    "German": 1.00, "French": 1.00, "Vietnamese": 1.00,   # dict-freq
    "Hindi": 1.00,                                        # mBERT hineng
}


def tagger_quality(first_language: str, second_language: str = "English") -> str:
    """``exact`` | ``provisional`` - how far to trust a row's tags.

    ``exact`` is correct by construction (script-disjoint). ``provisional``
    means measured at >=90% on a small self-labelled sample; human review is
    what promotes it to a defensible number.
    """
    if first_language in SCRIPT_DISJOINT_CONFIRMED:
        return "exact"
    return "provisional" if _MEASURED_ACCURACY.get(first_language, 0) >= 0.90 else "low"

In [ ]:
%%writefile src/tokenizer.py
"""Mixed-granularity tokenizer.

CJK does not tokenize on whitespace - a run of Han/Kana/Hangul between two
English words may hold several words with no space - so those scripts are cut
per character. Everything else is cut per word.

Two fixes over the pattern in the plan:

1. ``[A-Za-z]+`` splits accented Latin: "dificil" became ``dif|i|cil``, which
   shatters every Spanish/French/German/Vietnamese word carrying a diacritic
   and would badly inflate run counts. ``\\p{Script=Latin}`` covers the accented
   range, since U+00E9 and friends *are* Latin script.
2. Cyrillic, Arabic and Devanagari are space-delimited, so word-level is right
   for them; the plan's fallback branch would have cut them per character.
"""
import regex as re

# Order matters: character-level CJK alternatives are tried before the
# word-level ones, and the single-character fallback is last.
_PATTERN = re.compile(
    r"\p{Script=Han}"
    r"|\p{Script=Hiragana}"
    r"|\p{Script=Katakana}"
    r"|\p{Script=Hangul}"
    r"|[\p{Script=Latin}\p{M}]+(?:['’][\p{Script=Latin}\p{M}]+)*"
    r"|[\p{Script=Cyrillic}\p{M}]+"
    r"|[\p{Script=Arabic}\p{M}]+"
    r"|[\p{Script=Devanagari}\p{M}]+"
    r"|\d+"
    r"|[^\s]"
)


def tokenize_mixed(text: str) -> list[str]:
    """Split text into CJK-character tokens and word tokens for other scripts."""
    return _PATTERN.findall(text)

In [ ]:
%%writefile src/unicode_tagger.py
"""Script-based tagging, for language pairs whose scripts are disjoint.

Exact by construction: if the first language is written in Han and the second
in Latin, a token's script settles which language it belongs to. No model, no
threshold, nothing to validate beyond the script table itself.
"""
import regex as re

from .routing import expected_script

_SCRIPTS = ["Han", "Hiragana", "Katakana", "Hangul", "Devanagari",
            "Cyrillic", "Arabic", "Latin"]
_SCRIPT_RE = {name: re.compile(fr"\p{{Script={name}}}") for name in _SCRIPTS}


def char_script(ch: str) -> str:
    for name in _SCRIPTS:
        if _SCRIPT_RE[name].match(ch):
            return name
    return "other"


def tag_token_unicode(token: str) -> str:
    """Return the token's single script name, or ``"other"`` if mixed/unscripted.

    Combining marks are ignored when deciding, so an accented Latin token stays
    ``Latin`` rather than falling through to ``other``.
    """
    scripts = {char_script(c) for c in token if not c.isspace()}
    scripts.discard("other")
    if len(scripts) == 1:
        return scripts.pop()
    return "other"


def tag_tokens_unicode(tokens: list[str], first_language: str,
                       second_language: str) -> list[tuple[str, str]]:
    """Tag tokens as the row's first or second language by script.

    Returns ``(token, language)`` pairs, where language is the row's own
    ``first_language`` / ``second_language`` value, or ``"other"`` for
    punctuation and digits.
    """
    l1_scripts = expected_script(first_language)
    l2_scripts = expected_script(second_language)
    out = []
    for tok in tokens:
        script = tag_token_unicode(tok)
        if script in l1_scripts:
            out.append((tok, first_language))
        elif script in l2_scripts:
            out.append((tok, second_language))
        else:
            out.append((tok, "other"))
    return out

In [ ]:
%%writefile src/dict_tagger.py
"""Dictionary-lookup tagging - the obvious approach, measured.

Two variants of "look the word up in each language's dictionary":

``tag_naive``      if the word exists in the second language's list, call it the
                   second language. This is the intuitive rule, and it fails on
                   lexical overlap: most *common* words exist in both lists.
``tag_frequency``  compare how common the word is in each language and take the
                   higher. Strictly better, and the reason is instructive - the
                   question is never "is this word in the dictionary" but "which
                   language is it more characteristic of".

Frequencies come from `wordfreq` (Zipf scale: ~7 = very common, 0 = absent).
"""
from __future__ import annotations

from functools import lru_cache

from wordfreq import zipf_frequency

LANG_TO_CODE = {
    "English": "en", "Spanish": "es", "German": "de", "French": "fr",
    "Italian": "it", "Malay": "ms", "Philippines": "fil",
    "Vietnamese": "vi", "Hindi": "hi",
}


@lru_cache(maxsize=500_000)
def zipf(word: str, code: str) -> float:
    return zipf_frequency(word, code)


def _is_word(token: str) -> bool:
    return any(ch.isalpha() for ch in token)


def tag_naive(tokens: list[str], first_language: str,
              second_language: str) -> list[tuple[str, str]]:
    """"If it's in the second language's dictionary, it's the second language.\""""
    l2 = LANG_TO_CODE.get(second_language, "en")
    out = []
    for t in tokens:
        if not _is_word(t):
            out.append((t, "other"))
        elif zipf(t.lower(), l2) > 0:
            out.append((t, second_language))
        else:
            out.append((t, first_language))
    return out


def tag_frequency(tokens: list[str], first_language: str, second_language: str,
                  margin: float = 0.0) -> list[tuple[str, str]]:
    """Whichever language the word is more characteristic of wins."""
    l1 = LANG_TO_CODE.get(first_language)
    l2 = LANG_TO_CODE.get(second_language, "en")
    out = []
    for t in tokens:
        if not _is_word(t) or l1 is None:
            out.append((t, "other" if not _is_word(t) else first_language))
            continue
        w = t.lower()
        f1, f2 = zipf(w, l1), zipf(w, l2)
        if f1 == 0 and f2 == 0:
            out.append((t, "other"))          # unknown to both: proper noun etc.
        else:
            out.append((t, second_language if f2 > f1 + margin else first_language))
    return out


def overlap_stats(tokens: list[str], first_language: str,
                  second_language: str) -> dict:
    """How often is a word present in BOTH dictionaries? That is the ceiling
    on the naive rule: every such word is a coin flip for it."""
    l1 = LANG_TO_CODE.get(first_language)
    l2 = LANG_TO_CODE.get(second_language, "en")
    words = [t.lower() for t in tokens if _is_word(t)]
    if not words or l1 is None:
        return dict(words=0, both=0, neither=0, only_l1=0, only_l2=0)
    both = neither = only1 = only2 = 0
    for w in words:
        a, b = zipf(w, l1) > 0, zipf(w, l2) > 0
        both += a and b
        neither += not a and not b
        only1 += a and not b
        only2 += b and not a
    return dict(words=len(words), both=both, neither=neither,
                only_l1=only1, only_l2=only2)

In [ ]:
%%writefile src/lexical_override.py
"""Overrule AnE-LID where dict-freq's evidence is overwhelming (added 2026-11-03).

AnE-LID was chosen on LinCE spa-eng / hin-eng and Denglisch, the only
human-annotated sets we have. It was never checked on Malay, Vietnamese or
Tagalog, and there it tags common L1 function words English: Malay ``akan``
×1,496, ``dan``, ``yang``; Vietnamese ``và``, ``sẽ``, ``của``, even spans like
"bekerjasama dengan agensi lain untuk". Each one invents two switches.
``scripts/consensus_f1.py`` found 40-45% of the positions around such words
labelled as switches, and the trained model refusing them (recall 0.16-0.18).

Two rules, applied to word tokens that are not names:

1. **Vietnamese letters.** In a Vietnamese row, a word containing ``đ ă ơ ư``
   or a stacked tone mark (U+1EA0-U+1EF9) is Vietnamese. No English word has
   them; French/Spanish accents (é, ê, ñ) are not in the set.
2. **Frequency margin.** The word flips to the other language of the pair when
   it is common there (Zipf >= ``FLOOR``) and at least ``MARGIN`` Zipf more
   common there than in the language AnE gave it. Margin 2.0 = 100x more
   frequent. Shared words (``a``, ``in``, ``me``, ``online``) never clear it.

Scope: ``OVERRIDE_L1`` with English as L2. Romanized Hindi is excluded
(wordfreq's Hindi list is Devanagari, so dict-freq can't read it), German->French
is dict-freq already, and script-tagged pairs are exact.
"""
from __future__ import annotations

import unicodedata

from .dict_tagger import LANG_TO_CODE, zipf

OVERRIDE_L1 = frozenset({"Spanish", "Italian", "Malay", "Philippines", "German", "French",
                         "Vietnamese"})
MARGIN = 2.0
FLOOR = 4.0
_VI_LETTERS = frozenset("đĐăĂơƠưƯ")


def has_vietnamese_letters(token: str) -> bool:
    """True if the word has a letter only Vietnamese uses (checked after NFC)."""
    for ch in unicodedata.normalize("NFC", token):
        if ch in _VI_LETTERS or 0x1EA0 <= ord(ch) <= 0x1EF9:
            return True
    return False


def in_scope(first_language: str, second_language: str) -> bool:
    return second_language == "English" and first_language in OVERRIDE_L1


def override_tagged(tagged: list[tuple[str, str]], first_language: str, second_language: str,
                    is_entity: list[bool] | None = None, margin: float = MARGIN,
                    floor: float = FLOOR) -> tuple[list[tuple[str, str]], list[bool]]:
    """Return ``(tagged, changed)``: the corrected labels and which tokens moved."""
    if not in_scope(first_language, second_language):
        return list(tagged), [False] * len(tagged)
    l1, l2 = first_language, second_language
    c1, c2 = LANG_TO_CODE[l1], LANG_TO_CODE[l2]
    ents = is_entity if is_entity is not None else [False] * len(tagged)
    out, changed = [], []
    for (tok, lang), ent in zip(tagged, ents):
        new = lang
        if not ent and lang in (l1, l2) and any(ch.isalpha() for ch in tok):
            if l1 == "Vietnamese" and has_vietnamese_letters(tok):
                new = l1
            else:
                w = tok.lower()
                f1, f2 = zipf(w, c1), zipf(w, c2)
                if lang == l2 and f1 >= floor and f1 - f2 >= margin:
                    new = l1
                elif lang == l1 and f2 >= floor and f2 - f1 >= margin:
                    new = l2
        out.append((tok, new))
        changed.append(new != lang)
    return out, changed

In [ ]:
%%writefile src/ane_tagger.py
"""AnE-LID: one word-level tagger for every "<L1> vs English" pair.

``igorsterner/AnE-LID`` (Sterner, VarDial 2024) is XLM-R-large fine-tuned on
four English code-switch pairs at once - LinCE hineng/spaeng/nepeng plus
Denglisch - with the labels ``English`` / ``notEnglish`` / ``Mixed`` /
``Other``. Training on several pairs is what lets it transfer to pairs it never
saw, where the single-pair LinCE models (and our XLM-R hineng fine-tune) do
not. Licence: academic use only.

It only answers "English or not", so it cannot tag the German<->French rows.

The companion ``igorsterner/AnE-NER`` (same base, labels ``I`` / ``O``) marks
named-entity words; ``entity_mask_batch`` exposes it for the NE pass.

``Mixed`` (an English stem inside L1 morphology, "rewatchen") maps to the first
language - the word's grammatical host. ``Other`` maps to ``other``.

Word probabilities are the mean of the word's sub-token softmaxes, which is the
model card's majority vote made continuous, so a confidence comes for free.
"""
from __future__ import annotations

import sys

import torch

from .routing import ane_to_language as to_language  # noqa: F401  (public name)

REPO = "igorsterner/AnE-LID"
NER_REPO = "igorsterner/AnE-NER"
_REGISTRY = "_codeswitch_ane_registry"   # on `sys` for the same reason as bert_tagger
_CHUNK_WORDS = 170                       # headroom under 512 sub-tokens


def _device() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    return "cuda" if torch.cuda.is_available() else "cpu"


def load(repo: str = REPO, device: str | None = None):
    """Load (once per session) ``repo`` onto ``device`` (default: best available).

    On CUDA the model runs in fp16, which roughly doubles throughput; on MPS it
    measured no faster, and on CPU it is unsupported, so both stay fp32. Masks
    agreed with fp32 on 99.99% of tokens.
    """
    reg = getattr(sys, _REGISTRY, None)
    if reg is None:
        reg = {}
        setattr(sys, _REGISTRY, reg)
    dev = device or _device()
    key = (repo, dev)                  # one copy per GPU, so passes can shard
    if key not in reg:
        from transformers import AutoModelForTokenClassification, AutoTokenizer
        tok = AutoTokenizer.from_pretrained(repo)
        mdl = AutoModelForTokenClassification.from_pretrained(repo).eval()
        if dev.startswith("cuda"):
            mdl = mdl.half()
        reg[key] = (tok, mdl.to(dev))
    return reg[key]


def entity_mask_batch(sentences: list[list[str]],
                      device: str | None = None) -> list[list[bool]]:
    """True where AnE-NER calls the word part of a named entity."""
    return [[lab == "I" for lab, _ in row]
            for row in word_probs_batch(sentences, repo=NER_REPO, device=device)]


def word_probs_batch(sentences: list[list[str]], repo: str = REPO,
                     device: str | None = None) -> list[list[tuple[str, float]]]:
    """``(label, confidence)`` per word, for a batch of pre-tokenised sentences.

    Sentences longer than ``_CHUNK_WORDS`` are split into chunks, so no word is
    ever truncated away; context is preserved within a chunk.
    """
    tok, mdl = load(repo, device)
    chunks, owner = [], []
    for si, words in enumerate(sentences):
        for start in range(0, max(len(words), 1), _CHUNK_WORDS):
            chunks.append(list(words[start:start + _CHUNK_WORDS]))
            owner.append(si)

    enc = tok(chunks, is_split_into_words=True, return_tensors="pt",
              padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        probs = mdl(**{k: v.to(mdl.device) for k, v in enc.items()}).logits.float().softmax(-1).cpu()

    out: list[list[tuple[str, float]]] = [[] for _ in sentences]
    id2label = mdl.config.id2label
    for ci, words in enumerate(chunks):
        acc = [[] for _ in words]
        for pos, wid in enumerate(enc.word_ids(ci)):
            if wid is not None:
                acc[wid].append(probs[ci, pos])
        for a in acc:
            if not a:
                out[owner[ci]].append(("Other", 0.0))
                continue
            p = torch.stack(a).mean(0)
            out[owner[ci]].append((id2label[int(p.argmax())], float(p.max())))
    return out




def labels_batch(sentences: list[list[str]], device: str | None = None) -> list[list[str]]:
    """Raw AnE-LID label per word - what ``persist.ane_labels`` caches."""
    return [[lab for lab, _ in row] for row in word_probs_batch(sentences, device=device)]


def tag_tokens_ane(tokens: list[str], first_language: str,
                   second_language: str = "English",
                   labels: list[str] | None = None) -> list[tuple[str, str]]:
    """Same contract as the other taggers: ``(token, language)`` pairs.

    Pass ``labels`` (from :func:`labels_batch`) to skip the model call.
    """
    if labels is None or len(labels) != len(tokens):
        labels = labels_batch([tokens])[0]
    return [(t, "other" if not any(c.isalpha() for c in t)
             else to_language(lab, first_language, second_language))
            for t, lab in zip(tokens, labels)]

In [ ]:
%%writefile src/bert_tagger.py
"""Context-aware token classification via LinCE code-switch BERT models.

fastText scores each token in isolation, which is why it cannot resolve tokens
that are only disambiguated by their neighbours ("mas", "bila", "real"). These
are ``BertForTokenClassification`` models trained on the LinCE code-switch
benchmark, so they see the whole utterance.

Two models, routed by measured accuracy rather than by language family (see
``routing.MODEL_FOR``):

* ``hineng`` - the only thing that handles romanized Hinglish at all.
* ``spaeng`` - transfers to Italian/Malay/Tagalog as a generic
  Latin-L1-vs-English discriminator, but *regresses* German and French, which
  stay on fastText.

The ``ne`` (named-entity) label is mapped to ``other`` so proper nouns get
absorbed into the surrounding run instead of being forced into a language -
"Carney" and "Trump" are neither Hindi nor English.
"""
from __future__ import annotations

import sys
import time

import torch

from .routing import MODEL_FOR, MODEL_L1_LABEL, MODEL_REPO

_MAX_LEN = 512

# The registry lives on the `sys` module rather than here, deliberately.
# A module-level dict is wiped whenever IPython's %autoreload re-executes this
# file - which happens on *every* call in a notebook with autoreload on - so the
# 679 MB model would be reloaded from disk per row. `sys` is never reloaded, so
# hanging the cache there makes model loading genuinely once-per-session.
_REGISTRY = "_codeswitch_model_registry"


def _cache_dict() -> dict:
    reg = getattr(sys, _REGISTRY, None)
    if reg is None:
        reg = {}
        setattr(sys, _REGISTRY, reg)
    return reg


def _device() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    return "cuda" if torch.cuda.is_available() else "cpu"


def load(model_key: str, verbose: bool = True):
    """Load (once per session) a LinCE tokenizer/model pair onto the best device."""
    cache = _cache_dict()
    if model_key not in cache:
        from transformers import AutoModelForTokenClassification, AutoTokenizer
        repo = MODEL_REPO[model_key]
        t0 = time.time()
        tok = AutoTokenizer.from_pretrained(repo)
        mdl = AutoModelForTokenClassification.from_pretrained(repo).eval().to(_device())
        cache[model_key] = (tok, mdl)
        if verbose:
            print(f"[bert_tagger] loaded {model_key} on {_device()} "
                  f"in {time.time() - t0:.1f}s (once per session)", flush=True)
    return cache[model_key]


def diagnose() -> None:
    """Print why tagging might be slow. Run this if a cell takes minutes."""
    import transformers
    print(f"python      : {sys.executable}")
    print(f"transformers: {transformers.__version__}  (need >=5 with torch>=2.11)")
    print(f"torch       : {torch.__version__}")
    print(f"device      : {_device()}")
    cached = sorted(_cache_dict())
    print(f"models held : {cached or 'NONE - will load on first use'}")
    ip = sys.modules.get("IPython")
    if ip is not None and ip.get_ipython() is not None:
        exts = ip.get_ipython().extension_manager.loaded
        if "autoreload" in exts:
            print("autoreload  : ON - safe now (registry lives on sys), but it "
                  "wipes module globals; restart the kernel if you saw slowness")
        else:
            print("autoreload  : off")


def _label_words(model_key: str, words: list[str]) -> list[str]:
    """Raw LinCE label per word; first sub-token of each word decides."""
    tok, mdl = load(model_key)
    enc = tok(words, is_split_into_words=True, return_tensors="pt",
              truncation=True, max_length=_MAX_LEN)
    word_ids = enc.word_ids(0)
    enc = {k: v.to(mdl.device) for k, v in enc.items()}
    with torch.no_grad():
        pred = mdl(**enc).logits.argmax(-1)[0].tolist()
    out, seen = [], set()
    for pos, wid in enumerate(word_ids):
        if wid is not None and wid not in seen:
            seen.add(wid)
            out.append(mdl.config.id2label[pred[pos]])
    # Words past the 512-token limit are unlabelled; mark them unknown.
    return out + ["unk"] * (len(words) - len(out))


def tag_tokens_bert(tokens: list[str], first_language: str,
                    second_language: str) -> list[tuple[str, str]]:
    """Tag tokens with the LinCE model chosen for ``first_language``.

    Long token streams are processed in ``_MAX_LEN``-sized chunks so context is
    preserved within a chunk rather than lost per sentence.
    """
    key = MODEL_FOR[first_language]
    l1_label = MODEL_L1_LABEL[key]

    labels: list[str] = []
    # Leave headroom for [CLS]/[SEP] and sub-token expansion.
    step = _MAX_LEN // 3
    for start in range(0, len(tokens), step):
        labels.extend(_label_words(key, tokens[start:start + step]))

    out = []
    for tok, lab in zip(tokens, labels):
        if not any(ch.isalpha() for ch in tok):
            out.append((tok, "other"))
        elif lab == "en":
            out.append((tok, second_language))
        elif lab == l1_label:
            out.append((tok, first_language))
        else:
            # ne / ambiguous / mixed / fw / other / unk -> absorbed downstream.
            out.append((tok, "other"))
    return out

In [ ]:
%%writefile src/dispatch.py
"""Single entry point that routes a token stream to the right tagger.

Kept separate so ``segments``, ``inspection`` and ``persist`` share one copy of
the branch, and so the heavy imports (torch, the models) stay lazy - a
unicode-routed row loads none of them.
"""
from __future__ import annotations

import regex as re

from .routing import ane_to_language, route

_DEVANAGARI = re.compile(r"\p{Script=Devanagari}")
_LATIN = re.compile(r"\p{Script=Latin}")


def _hindi_is_devanagari(tokens: list[str]) -> bool:
    """Hindi rows come in two scripts; only the romanized half needs a model.

    Routing per *language* sent Devanagari rows to the LinCE model, which is
    trained on romanized Hinglish and so is out of distribution for them - while
    the Unicode tagger handles Devanagari-vs-Latin exactly. Decide per row.
    """
    dev = sum(1 for t in tokens if _DEVANAGARI.search(t))
    lat = sum(1 for t in tokens if _LATIN.search(t))
    return dev > lat


def row_strategy(tokens: list[str], first_language: str, second_language: str) -> str:
    """``route`` plus the per-row Devanagari check for Hindi."""
    strategy = route(first_language, second_language)
    if (strategy in ("ane", "bert") and first_language == "Hindi"
            and _hindi_is_devanagari(tokens)):
        return "unicode"
    return strategy


def _devanagari_is_hindi(tagged, first_language):
    """Romanized-majority Hindi rows still carry some Devanagari words, and the
    LinCE model called 24,113 of them English. Script settles those exactly."""
    if first_language != "Hindi":
        return tagged
    return [(t, first_language if _DEVANAGARI.search(t) else lang) for t, lang in tagged]


def _names_from_context(tagged, is_entity, ane_labels, first_language, second_language):
    """Script-tagged rows: a Latin-script *name* takes AnE-LID's contextual label.

    Script alone calls every Latin token English, which is right for
    "Game of Thrones" but invents a switch for "Messi" in a Cantonese sentence.
    AnE-LID, run on a window around the name, tells the two apart. Names in the
    row's own script (中關村) stay L1, and non-entity tokens keep their script
    label. An empty label means the name was never scored; script stands.
    """
    out = []
    for (t, lang), ent, lab in zip(tagged, is_entity, ane_labels):
        if ent and lab and _LATIN.search(t):
            lang = ane_to_language(lab, first_language, second_language)
        out.append((t, lang))
    return out


def tag_tokens(tokens: list[str], first_language: str, second_language: str,
               ane_labels: list[str] | None = None,
               is_entity: list[bool] | None = None) -> list[tuple[str, str]]:
    """Tag tokens as first/second language via whichever tier the pair needs.

    ``ane_labels`` are precomputed raw AnE-LID word labels (``persist.ane_labels``
    batches them over the corpus). For ``ane`` rows they are the labels; without
    them the model runs on this row. For script-tagged rows they are optional
    and only cover the windows around Latin-script names (see
    ``_names_from_context``); ``""`` marks unscored positions.
    """
    strategy = row_strategy(tokens, first_language, second_language)
    if strategy == "ane":
        from .ane_tagger import tag_tokens_ane
        from .lexical_override import override_tagged
        tagged = _devanagari_is_hindi(
            tag_tokens_ane(tokens, first_language, second_language, labels=ane_labels),
            first_language)
        ents = is_entity if is_entity is not None and len(is_entity) == len(tagged) else None
        return override_tagged(tagged, first_language, second_language, ents)[0]
    if strategy == "unicode":
        from .unicode_tagger import tag_tokens_unicode
        tagged = tag_tokens_unicode(tokens, first_language, second_language)
        if (ane_labels is not None and is_entity is not None
                and len(ane_labels) == len(tokens) == len(is_entity)):
            tagged = _names_from_context(tagged, is_entity, ane_labels,
                                         first_language, second_language)
        return tagged
    if strategy == "bert":
        from .bert_tagger import tag_tokens_bert
        return _devanagari_is_hindi(
            tag_tokens_bert(tokens, first_language, second_language), first_language)
    if strategy == "dict":
        from .dict_tagger import tag_frequency
        return tag_frequency(tokens, first_language, second_language)
    from .model_tagger import tag_tokens_model
    return tag_tokens_model(tokens, first_language, second_language)

In [ ]:
%%writefile src/segments.py
"""Run-length encoding of tagged tokens, plus the dataset builders.

The plan leaves one decision open (line 174): whether an ``"other"`` tag
(punctuation, digits) breaks a run or is absorbed into the surrounding
language. Absorbing is cleaner for the downstream goals, so ``absorb_other``
does that - with one guard the naive version gets wrong: an ``other`` token
sitting *between two different languages* must not bridge them into a single
run. "hola, hello" is two runs, not one.

Named entities: every token keeps a *language* label, and entity-ness travels
separately as an ``is_entity`` flag. How names count is a policy, not a label
(``resolve_names``; every edge case is pinned in tests/test_name_strategies.py).
AnE-NER fragments names, so ``repair_spans`` mends them first for every policy.

* ``names="script_words"`` (default) - a name is a switch only with visible
  evidence. A Latin-script name in a non-Latin row is English ("Messi",
  "Severance Season 2" inside Arabic), since the writer chose the Latin script;
  a name in the row's own script stays L1. In Latin-script rows the name's
  words decide when they are clearly one language's ("Severance Season" ->
  English, "Union europeenne" -> French); otherwise it takes its surroundings
  ("Ayer Leo Messi marco" -> Spanish), and may not open a run on its own.
* ``names="transparent"`` - names are no language and are skipped (LinCE's
  convention). Stored as the main ablation.
* ``names="script"`` - every Latin-script name is English. Stored too; adds a
  one-word switch for each Messi / Monza / EU inside a Latin-script sentence.
* ``names="hybrid"`` - titles keep their language, other names take context.
* ``names="language"`` - names count as AnE-LID tagged them.
"""
from __future__ import annotations

import ast

import regex as re

from .routing import route
from .tokenizer import tokenize_mixed

NE = "NE"
NAME_POLICIES = ("hybrid", "language", "transparent", "script", "script_words")
# A name counts as a switch only with visible evidence: a change of script, or
# words that are clearly one language's. See tests/test_name_strategies.py.
DEFAULT_NAMES = "script_words"

# A function word: short and extremely frequent in its language (wordfreq Zipf).
# Measured: of/the/and/de/la/der/dan/ng/cua all score 6.5-7.9; the most common
# content words in candidate names (game, real, united, league) stay <= 5.8.
_FUNCTION_ZIPF = 6.4
_FUNCTION_MAXLEN = 4


def _is_function_word(word: str, language: str) -> bool:
    from .dict_tagger import LANG_TO_CODE, zipf
    code = LANG_TO_CODE.get(language)
    return (code is not None and len(word) <= _FUNCTION_MAXLEN
            and zipf(word.lower(), code) >= _FUNCTION_ZIPF)


_LATIN = re.compile(r"\p{Script=Latin}")
# Languages whose wordfreq list is written in Latin script, so a Latin-script
# name can be looked up in it. Hindi's list is Devanagari: romanized names
# score ~0 there, which would make every Bollywood name "English".
_LATIN_DICT = {"English", "Spanish", "German", "French", "Italian", "Malay",
               "Philippines", "Vietnamese"}
_MARGIN = 1.0      # Zipf gap for a word to count as clearly one language's


def _latin(tok: str) -> bool:
    return bool(_LATIN.search(tok))


def _zipf(word: str, language: str):
    if language not in _LATIN_DICT:
        return None
    from .dict_tagger import LANG_TO_CODE, zipf
    return zipf(word.lower(), LANG_TO_CODE[language])


def _word_side(word: str, first_language: str, second_language: str):
    """Which of the pair a Latin word clearly belongs to, by dictionary margin; None if unclear.

    Names sit in both frequency lists at similar levels (messi 3.8 en / 4.1 es,
    madrid 4.3 / 5.4), so only a >= 1.0 Zipf gap decides: severance, season ->
    English; europeenne, monza -> L1; messi, psg, eu -> unclear.
    """
    z1, z2 = _zipf(word, first_language), _zipf(word, second_language)
    if z1 is None or z2 is None:
        return None
    if z2 - z1 >= _MARGIN:
        return second_language
    if z1 - z2 >= _MARGIN:
        return first_language
    return None


def repair_spans(tokens, is_entity, first_language: str, second_language: str) -> list[bool]:
    """Mend AnE-NER's fragmented Latin-script names before any policy uses them.

    Measured on real output: "Severance Season 2" came back as just
    "Severance", "Game of Thrones" as "Game" + "Thrones", "The University of
    Texas" as "of Texas". Two repairs, Latin script only:

    * bridge a single function word between two name words ("Game [of] Thrones");
    * grow a name over adjacent capitalised words - and numbers to its right -
      unless the word is clearly an ordinary L1 word ("Ayer Leo Messi" keeps
      "Ayer"; German nouns like "Kommission" are never swallowed).
    Non-Latin names are left exactly as tagged.
    """
    mask = list(is_entity)
    n = len(tokens)
    for k in range(1, n - 1):
        if (not mask[k] and mask[k - 1] and mask[k + 1]
                and _latin(tokens[k - 1]) and _latin(tokens[k + 1])
                and any(_is_function_word(tokens[k], lang)
                        for lang in {first_language, second_language, "English"})):
            mask[k] = True

    def joins(tok: str, to_the_right: bool) -> bool:
        if to_the_right and tok.isdigit():
            return True
        return (_latin(tok) and tok[:1].isupper()
                and _word_side(tok, first_language, second_language) != first_language)

    changed = True
    while changed:
        changed = False
        for k in range(n):
            if mask[k]:
                continue
            after_name = k > 0 and mask[k - 1] and (_latin(tokens[k - 1]) or tokens[k - 1].isdigit())
            before_name = k + 1 < n and mask[k + 1] and _latin(tokens[k + 1])
            if (after_name and joins(tokens[k], True)) or (before_name and joins(tokens[k], False)):
                mask[k] = changed = True
    return mask


def _context_language(out, ent, i, j, own, langs):
    """An opaque name (tokens i..j-1) may not open a run of its own.

    Inside one language it takes that language. At a real switch it joins the
    side not cut off by punctuation; if neither or both are, it keeps ``own``
    when that is one of the two sides.
    """
    n = len(out)

    def neighbour(k, step):
        k += step
        cut = False
        while 0 <= k < n:
            t, lang = out[k]
            if not _is_word(t):
                cut = True
            elif not ent[k] and lang in langs:
                return lang, cut
            k += step
        return None, cut

    (prev, prev_cut), (nxt, nxt_cut) = neighbour(i, -1), neighbour(j - 1, 1)
    if prev and nxt and prev != nxt:
        if prev_cut != nxt_cut:
            return nxt if prev_cut else prev
        return own if own in (prev, nxt) else prev
    return prev or nxt or own


def _spans(ent):
    i, n = 0, len(ent)
    while i < n:
        if not ent[i]:
            i += 1
            continue
        j = i
        while j < n and ent[j]:
            j += 1
        yield i, j
        i = j


def resolve_names(tagged: list[tuple[str, str]], is_entity, first_language: str,
                  second_language: str, names: str = DEFAULT_NAMES) -> list[tuple[str, str]]:
    """Apply a names policy to language-tagged tokens (see the module docstring)."""
    if is_entity is None or names == "language":
        return list(tagged)
    if names not in NAME_POLICIES:
        raise ValueError(f"unknown names policy {names!r}; use one of {NAME_POLICIES}")
    toks = [t for t, _ in tagged]
    ent = repair_spans(toks, is_entity, first_language, second_language)
    if names == "transparent":
        return apply_entities(tagged, ent)

    from .dispatch import _hindi_is_devanagari
    from .routing import SCRIPT_DISJOINT_CONFIRMED
    script_row = (first_language in SCRIPT_DISJOINT_CONFIRMED
                  or (first_language == "Hindi" and _hindi_is_devanagari(toks)))
    langs = (first_language, second_language)
    out = list(tagged)
    for i, j in _spans(ent):
        words = [k for k in range(i, j) if _is_word(out[k][0])]
        tagged_langs = [out[k][1] for k in words if out[k][1] in langs]
        if not tagged_langs:
            continue
        own = max(langs, key=tagged_langs.count)
        latin = any(_latin(out[k][0]) for k in words)

        if names == "hybrid":
            title = len(words) >= 2 and any(_is_function_word(out[k][0], own) for k in words)
            if not title:
                own = _context_language(out, ent, i, j, own, langs)
        elif names == "script":
            if latin and second_language == "English":
                own = second_language
        elif names == "script_words" and latin:
            if script_row:
                if second_language == "English":
                    own = second_language
            else:
                votes = [_word_side(out[k][0], first_language, second_language) for k in words]
                votes = [v for v in votes if v]
                if votes and votes.count(langs[0]) != votes.count(langs[1]):
                    own = max(langs, key=votes.count)
                else:
                    own = _context_language(out, ent, i, j, own, langs)
        for k in words:
            out[k] = (out[k][0], own)
    return out

def apply_entities(tagged: list[tuple[str, str]],
                   is_entity: list[bool]) -> list[tuple[str, str]]:
    """The ``transparent`` policy: relabel entity *words* ``NE``.

    Punctuation inside a name stays ``other``.
    """
    return [(t, NE if ent and _is_word(t) else lang)
            for (t, lang), ent in zip(tagged, is_entity)]


def absorb_other(tagged: list[tuple[str, str]]) -> list[tuple[str, str]]:
    """Reassign ``other`` tokens to the surrounding language.

    An ``other`` run takes the language of the token before it, unless that
    would merge two different languages - at a genuine switch boundary the
    punctuation joins the *following* language instead, so the boundary stays
    where it is. Leading ``other`` tokens join whatever follows. Token count is
    preserved.
    """
    out = list(tagged)
    n = len(out)
    i = 0
    while i < n:
        if out[i][1] != "other":
            i += 1
            continue
        j = i
        while j < n and out[j][1] == "other":
            j += 1
        # Look through NE tokens: they belong to neither neighbour, so the
        # boundary test must be made against the nearest real language.
        p = i - 1
        while p >= 0 and out[p][1] == NE:
            p -= 1
        q = j
        while q < n and out[q][1] in ("other", NE):
            q += 1
        prev_lang = out[p][1] if p >= 0 else None
        next_lang = out[q][1] if q < n else None
        if prev_lang is None:
            fill = next_lang
        elif next_lang is None or prev_lang == next_lang:
            fill = prev_lang
        else:
            # Genuine switch boundary: do not let punctuation bridge it.
            fill = next_lang
        if fill is not None:
            for k in range(i, j):
                out[k] = (out[k][0], fill)
        i = j
    return out


def to_segments(tagged_tokens: list[tuple[str, str]]) -> list[list]:
    """Collapse consecutive same-language tokens into ``[language, length]`` runs."""
    segments: list[list] = []
    for _, lang in tagged_tokens:
        if segments and segments[-1][0] == lang:
            segments[-1][1] += 1
        else:
            segments.append([lang, 1])
    return segments


def l2_runs(segments: list[list], second_language: str) -> list[int]:
    """Run lengths straight off the segments, punctuation included.

    Prefer :func:`l2_word_runs` for the switch-length question - segments carry
    absorbed punctuation, which inflates every run.
    """
    return [n for lang, n in segments if lang == second_language]


def _is_word(token: str) -> bool:
    return any(ch.isalpha() for ch in token)


def l2_word_runs(tagged: list[tuple[str, str]], second_language: str) -> list[int]:
    """Lengths of each consecutive second-language run, counting WORDS only.

    This is the original question: how many second-language tokens are used
    consecutively before switching back or ending.

    Punctuation is *transparent* - it neither breaks a run nor counts toward
    it. That matters because ``absorb_other`` hands boundary punctuation to the
    following language, so a sentence-final "." belonging to the first-language
    sentence would otherwise be counted as a second-language token and inflate
    every run by one.
    """
    runs, current = [], 0
    for token, lang in tagged:
        if not _is_word(token) or lang == NE:
            continue                      # transparent: no break, no count
        if lang == second_language:
            current += 1
        elif current:
            runs.append(current)
            current = 0
    if current:
        runs.append(current)
    return runs


def parse_utterances(cell: str) -> list[str]:
    """``data_generation_result`` is a stringified Python list of utterances."""
    try:
        value = ast.literal_eval(str(cell))
    except (ValueError, SyntaxError):
        return [str(cell)]
    if isinstance(value, list):
        return [str(v) for v in value]
    return [str(value)]


def tag_row(cell: str, first_language: str, second_language: str,
            cross_utterance: bool = True, is_entity: list[bool] | None = None,
            return_tagged: bool = False, ane_labels: list[str] | None = None,
            names: str = DEFAULT_NAMES) -> dict:
    """Tokenise, tag and run-length-encode one dataframe row.

    ``cross_utterance=True`` streams every utterance together, so a run may span
    a turn boundary; the utterance each run starts in is still recorded.
    """
    from .dispatch import row_strategy, tag_tokens

    utterances = parse_utterances(cell)

    tokens: list[str] = []
    origin: list[int] = []
    for idx, utt in enumerate(utterances):
        toks = tokenize_mixed(utt)
        tokens.extend(toks)
        origin.extend([idx] * len(toks))

    if not tokens:
        return dict(segments=[], l2_runs=[], n_switches=0, max_run=0,
                    total_l2_tokens=0, total_tokens=0, l2_token_ratio=0.0,
                    method=route(first_language, second_language),
                    unknown_rate=0.0, run_utterance_idx=[])

    strategy = row_strategy(tokens, first_language, second_language)
    tagged = tag_tokens(tokens, first_language, second_language,
                        ane_labels=ane_labels, is_entity=is_entity)
    lang_tagged = tagged                  # stored labels: always a language, policy-free
    tagged = resolve_names(tagged, is_entity, first_language, second_language, names)
    if strategy == "fasttext":
        from .model_tagger import unknown_rate
        unk = unknown_rate(tokens, first_language, second_language)
    else:
        unk = 0.0

    if not cross_utterance:
        # Break runs at turn boundaries by inserting a sentinel language.
        tagged = [
            t if i == 0 or origin[i] == origin[i - 1] else (t[0], t[1])
            for i, t in enumerate(tagged)
        ]

    tagged = absorb_other(tagged)
    segments = to_segments(tagged)
    runs = l2_word_runs(tagged, second_language)

    # Utterance index where each second-language run begins.
    run_start_idx: list[int] = []
    pos = 0
    for lang, n in segments:
        if lang == second_language:
            run_start_idx.append(origin[pos] if pos < len(origin) else -1)
        pos += n

    total_l2 = sum(runs)
    counted = sum(n for lang, n in segments if lang not in ("other", NE))
    result = dict(
        segments=segments,
        l2_runs=runs,
        n_switches=len(runs),
        max_run=max(runs) if runs else 0,
        total_l2_tokens=total_l2,
        total_tokens=len(tokens),
        l2_token_ratio=(total_l2 / counted) if counted else 0.0,
        method=strategy,
        unknown_rate=unk,
        run_utterance_idx=run_start_idx,
    )
    if return_tagged:
        result["tagged"] = lang_tagged   # pre-absorption, never NE
    return result

In [ ]:
%%writefile src/persist.py
"""Full-dataset tagging run and the parquet handoff for Goal 1 / Goal 2.

Nested Python objects do not round-trip cleanly through parquet, so the two
ragged structures are stored as parallel primitive lists rather than lists of
tuples or mixed-type pairs:

    tokens        list[str]   every token, in order
    token_lang    list[str]   its tagged language, same length as tokens
    segment_lang  list[str]   run-length encoding: the language of each run
    segment_len   list[int]   ... and its length, same length as segment_lang

Both pairs are parquet-native (``list<string>`` / ``list<int32>``), lossless,
and zip straight back together - see ``load_tagged``.
"""
from __future__ import annotations

import os

import pandas as pd

from .routing import route, tagger_quality
from .segments import parse_utterances, tag_row
from .tokenizer import tokenize_mixed

DATA_DIR = os.path.join(os.path.dirname(os.path.dirname(__file__)), "data")
DEFAULT_PATH = os.path.join(DATA_DIR, "tagged.parquet")
ENTITY_PATH = os.path.join(DATA_DIR, "entities.parquet")
ANE_LID_PATH = os.path.join(DATA_DIR, "ane_lid.parquet")

# Kept out of the parquet: the source text is already in the CSVs, and copying
# it doubles the file for no gain. Join back on `row_id` if you need it.
_KEEP = ["first_language", "second_language", "topic", "cs_ratio", "cs_type",
         "cs_function", "conversation_type", "score"]


def _tag_with_tokens(cell, first_language: str, second_language: str,
                     is_entity: list[bool] | None = None,
                     ane_labels: list[str] | None = None) -> dict:
    """``tag_row`` plus the flat token/language sequences Goal 1 needs.

    ``l2_runs`` & co. follow the default names policy (``script_words``);
    ``l2_runs_transparent`` and ``l2_runs_script`` are the stored ablations.
    ``token_lang`` itself is policy-free: the tagger's language for every
    word, names included.
    """
    result = tag_row(cell, first_language, second_language, is_entity=is_entity,
                     return_tagged=True, ane_labels=ane_labels)
    tagged = result.pop("tagged", [])
    result["tokens"] = [t for t, _ in tagged]
    result["token_lang"] = [l for _, l in tagged]
    mask = list(is_entity) if is_entity is not None else [False] * len(tagged)
    result["is_entity"] = mask
    result.update(_policy_runs(tagged, mask, first_language, second_language))
    result["segment_lang"] = [s[0] for s in result["segments"]]
    result["segment_len"] = [int(s[1]) for s in result["segments"]]
    del result["segments"]
    return result


def _policy_runs(tagged, mask, l1: str, l2: str) -> dict:
    from .segments import absorb_other, l2_word_runs, resolve_names
    return {f"l2_runs_{col}": l2_word_runs(absorb_other(resolve_names(tagged, mask, l1, l2, pol)), l2)
            for col, pol in (("transparent", "transparent"), ("script", "script"))}


def derive_name_policies(frame: pd.DataFrame) -> pd.DataFrame:
    """Recompute every names-policy column from ``tokens``/``token_lang``/``is_entity``.

    No model is involved, so this takes about a minute for the corpus. The
    default-policy columns - l2_runs, n_switches, max_run, total_l2_tokens,
    segment_lang/len - are overwritten; token_lang is left untouched.
    """
    from .segments import DEFAULT_NAMES, absorb_other, l2_word_runs, resolve_names, to_segments

    out = frame.copy()
    cols: dict[str, list] = {k: [] for k in ("l2_runs", "l2_runs_transparent", "l2_runs_script",
                                            "segment_lang", "segment_len")}
    for t, l, e, l1, l2 in zip(out.tokens, out.token_lang, out.is_entity,
                               out.first_language, out.second_language):
        tagged, mask = list(zip(t, l)), list(e)
        hyb = absorb_other(resolve_names(tagged, mask, l1, l2, DEFAULT_NAMES))
        cols["l2_runs"].append(l2_word_runs(hyb, l2))
        segs = to_segments(hyb)
        cols["segment_lang"].append([s[0] for s in segs])
        cols["segment_len"].append([int(s[1]) for s in segs])
        for k, v in _policy_runs(tagged, mask, l1, l2).items():
            cols[k].append(v)
    for k, v in cols.items():
        out[k] = v
    out["n_switches"] = [len(r) for r in out.l2_runs]
    out["max_run"] = [max(r) if len(r) else 0 for r in out.l2_runs]
    out["total_l2_tokens"] = [int(sum(r)) for r in out.l2_runs]
    return out


def apply_lexical_override(frame: pd.DataFrame) -> pd.DataFrame:
    """Apply ``lexical_override`` to the ``ane`` rows, then re-derive every policy column.

    Adds ``lexical_override``: per token, True where the label was changed, so
    the AnE-LID original is recoverable (flip it back to the pair's other
    language). Rows of other methods get an all-False mask.
    """
    from .lexical_override import override_tagged

    out = frame.copy()
    langs, masks = [], []
    for toks, labs, ents, l1, l2, method in zip(out.tokens, out.token_lang, out.is_entity,
                                                out.first_language, out.second_language,
                                                out.method):
        toks, labs = list(toks), list(labs)
        if method == "ane":
            tagged, changed = override_tagged(list(zip(toks, labs)), l1, l2, list(ents))
            labs = [l for _, l in tagged]
        else:
            changed = [False] * len(toks)
        langs.append(labs)
        masks.append(changed)
    out["token_lang"], out["lexical_override"] = langs, masks
    return derive_name_policies(out)


def from_v1(frame: pd.DataFrame) -> pd.DataFrame:
    """Turn v1 labels (names as ``NE``) into policy-free labels + ``is_entity``.

    v1 kept AnE-NER's name positions but not a language for the names, so one
    is reconstructed: the script tagger's label in script-tagged rows, dict-freq
    in the rest. The default policy barely consults it - in non-Latin rows a
    name's language comes from its script, in Latin rows from its words or its
    neighbours - it only decides a name sitting on an unpunctuated switch.
    Every non-name token is unchanged. Then all policy columns are derived.
    """
    from .dict_tagger import tag_frequency
    from .segments import NE
    from .unicode_tagger import tag_tokens_unicode

    out = frame.copy()
    langs, ents = [], []
    for toks, labs, l1, l2, method in zip(out.tokens, out.token_lang, out.first_language,
                                          out.second_language, out.method):
        toks, labs = list(toks), list(labs)
        ent = [x == NE for x in labs]
        if any(ent):
            fill = (tag_tokens_unicode(toks, l1, l2) if method == "unicode"
                    else tag_frequency(toks, l1, l2))
            labs = [f if e else x for x, (_, f), e in zip(labs, fill, ent)]
        langs.append(labs)
        ents.append(ent)
    out["token_lang"], out["is_entity"] = langs, ents
    return derive_name_policies(out)


def row_tokens(cell) -> list[str]:
    """The token sequence ``tag_row`` will see for one row."""
    tokens: list[str] = []
    for utt in parse_utterances(cell):
        tokens.extend(tokenize_mixed(utt))
    return tokens


def _cached_pass(df: pd.DataFrame, cache: str, column: str, fn, desc: str,
                 batch: int = 32, progress: bool = True,
                 select=None) -> dict[int, list]:
    """Run a batched per-word model ``fn`` over rows, cached by ``row_id``.

    ``fn`` takes ``[(row_id, tokens), ...]`` and returns one per-token list per
    row. ``row_id`` is ``df``'s index. Rows already cached are skipped, so an
    interrupted run resumes; a cached entry whose length no longer matches the
    row's tokens (tokenizer change) is recomputed. ``select(row, tokens)``
    restricts which rows need the pass at all.
    """
    os.makedirs(os.path.dirname(cache) or ".", exist_ok=True)
    done: dict[int, list] = {}
    if os.path.exists(cache):
        c = pd.read_parquet(cache)
        done = {int(r): list(m) for r, m in zip(c.row_id, c[column])}

    todo = []
    for row in df.itertuples():
        toks = row_tokens(row.data_generation_result)
        if select is not None and not select(row, toks):
            continue
        if len(done.get(int(row.Index), ())) != len(toks):
            todo.append((int(row.Index), toks))

    bar = None
    if progress and todo:
        try:
            from tqdm.auto import tqdm
            bar = tqdm(total=len(todo), desc=desc)
        except ImportError:
            pass
    # Sort by length so each batch pads to similar sizes; flush every ~2k rows.
    todo.sort(key=lambda x: len(x[1]))
    since_flush = 0
    for b in range(0, len(todo), batch):
        chunk = todo[b:b + batch]
        for (rid, _), out in zip(chunk, fn(chunk)):
            done[rid] = out
        since_flush += len(chunk)
        if bar:
            bar.update(len(chunk))
        if since_flush >= 2048 or b + batch >= len(todo):
            pd.DataFrame({"row_id": list(done), column: list(done.values())}
                         ).to_parquet(cache, index=False)
            since_flush = 0
    if bar:
        bar.close()
    return done


def entity_masks(df: pd.DataFrame, cache: str = ENTITY_PATH, device: str | None = None,
                 **kw) -> dict[int, list[bool]]:
    """AnE-NER mask per row, cached in ``data/entities.parquet``.

    This is the one expensive model pass *every* row needs (~8 h on MPS), so it
    is stored separately: a tagger or routing change re-tags without redoing it.
    """
    from .ane_tagger import entity_mask_batch
    return _cached_pass(df, cache, "is_entity",
                        lambda items: entity_mask_batch([t for _, t in items], device=device),
                        "entities", **kw)


def _name_windows(tokens: list[str], mask: list[bool], window: int) -> list[tuple[int, int]]:
    """Merged ``[start, end)`` spans of +-``window`` tokens around Latin-script names."""
    import regex as re
    latin = re.compile(r"\p{Script=Latin}")
    spans: list[list[int]] = []
    for k, (t, ent) in enumerate(zip(tokens, mask)):
        if ent and latin.search(t):
            a, b = max(0, k - window), min(len(tokens), k + window + 1)
            if spans and a <= spans[-1][1]:
                spans[-1][1] = max(spans[-1][1], b)
            else:
                spans.append([a, b])
    return [(a, b) for a, b in spans]


def ane_labels(df: pd.DataFrame, cache: str = ANE_LID_PATH, device: str | None = None,
               entities: dict[int, list[bool]] | None = None, window: int = 16,
               **kw) -> dict[int, list[str]]:
    """Raw AnE-LID word labels, cached on disk.

    * ``ane`` rows: every token (these labels *are* the row's tags).
    * script-tagged rows with a Latin-script entity (needs ``entities``): only
      +-``window`` tokens around each such name, so its label comes from real
      context without paying for the whole row; other positions are ``""``.
      This is what lets "Messi" in a Cantonese sentence stop being English.
    """
    from .ane_tagger import labels_batch
    from .dispatch import row_strategy

    ents = entities or {}
    windows: dict[int, list[tuple[int, int]]] = {}   # script rows -> name windows

    def needs(row, toks):
        s = row_strategy(toks, row.first_language, row.second_language)
        if s == "ane":
            return True
        m = ents.get(int(row.Index))
        if s != "unicode" or m is None:
            return False
        spans = _name_windows(toks, m, window)
        if spans:
            windows[int(row.Index)] = spans
        return bool(spans)

    def fn(items):
        jobs, where = [], []           # one model input per full row or window
        out = []
        for i, (rid, toks) in enumerate(items):
            for a, b in windows.get(rid, [(0, len(toks))]):   # ane rows: whole row
                jobs.append(toks[a:b])
                where.append((i, a))
            out.append([""] * len(toks))
        for (i, a), labs in zip(where, labels_batch(jobs, device=device)):
            out[i][a:a + len(labs)] = labs
        return out

    return _cached_pass(df, cache, "ane_label", fn, "ane-lid", select=needs, **kw)


def tag_dataframe(df: pd.DataFrame, progress: bool = True,
                  entities: dict[int, list[bool]] | None = None,
                  ane: dict[int, list[str]] | None = None) -> pd.DataFrame:
    """Tag every row. Returns a frame aligned to ``df`` by position.

    ``entities`` (from :func:`entity_masks`) relabels named-entity words ``NE``;
    without it, names get whatever language the tagger assigned. ``ane`` (from
    :func:`ane_labels`) supplies precomputed AnE-LID labels; rows missing from
    it run the model inline, one row at a time.
    """
    rows = df.itertuples()
    if progress:
        try:
            from tqdm.auto import tqdm
            rows = tqdm(rows, total=len(df), desc="tagging")
        except ImportError:
            pass

    ents, lid = entities or {}, ane or {}
    out = pd.DataFrame([
        _tag_with_tokens(r.data_generation_result, r.first_language, r.second_language,
                         ents.get(int(r.Index)), lid.get(int(r.Index)))
        for r in rows
    ])
    out.insert(0, "row_id", df.index.to_numpy())
    for col in _KEEP:
        if col in df.columns:
            out[col] = df[col].to_numpy()
    out["quality"] = [tagger_quality(a, b) for a, b in
                      zip(out.first_language, out.second_language)]
    return out


# One code per token in the CSV export. Language names vary per row, so the
# codes are relative to the row's own pair - the same binarisation the
# anticipation task uses. Names carry a language code like any other word,
# resolved with the default hybrid policy; entity-ness is the ``entities`` column.
LABEL_CODES = "L1 = first_language, L2 = second_language, O = punctuation/digits/other"


def _code(lang: str, l1: str, l2: str) -> str:
    if lang == l1:
        return "L1"
    if lang == l2:
        return "L2"
    return "O"


def export_rows_csv(frame: pd.DataFrame, path: str) -> str:
    """One CSV line per corpus row, tokens and labels as aligned space-joined strings.

    Tokens never contain whitespace (see ``tokenizer``), so
    ``row.tokens.split(" ")`` and ``row.labels.split(" ")`` zip back together.
    ``entities`` is an aligned 0/1 string (1 = part of a named entity).
    ``labels`` and ``l2_runs`` follow the default names policy (script_words);
    ``l2_runs`` are the second-language run lengths in words - the "how long
    does a switch last" targets - with punctuation transparent.
    ``l2_runs_names_transparent`` / ``l2_runs_names_script`` are the stored
    alternatives (see ``segments.resolve_names``).
    ``split`` is the row-level train/val/test assignment used everywhere else
    (``row_id % 10``: 0-7 train, 8 val, 9 test).
    """
    from .segments import resolve_names

    split = {**{i: "train" for i in range(8)}, 8: "val", 9: "test"}
    ents = (list(frame.is_entity) if "is_entity" in frame.columns
            else [[False] * len(t) for t in frame.tokens])
    out = pd.DataFrame({
        "row_id": frame.row_id,
        "first_language": frame.first_language,
        "second_language": frame.second_language,
        "split": [split[int(r) % 10] for r in frame.row_id],
        "quality": frame.quality,
        "method": frame.method,
        "n_tokens": [len(t) for t in frame.tokens],
        "tokens": [" ".join(t) for t in frame.tokens],
        "labels": [" ".join(_code(l, a, b) for _, l in resolve_names(list(zip(t, ls)), e, a, b))
                   for t, ls, e, a, b in zip(frame.tokens, frame.token_lang, ents,
                                             frame.first_language, frame.second_language)],
        "entities": [" ".join("1" if e else "0" for e in m) for m in ents],
        "l2_runs": [" ".join(str(int(x)) for x in r) for r in frame.l2_runs],
        "l2_runs_names_script": [
            " ".join(str(int(x)) for x in r) for r in
            (frame.l2_runs_script if "l2_runs_script" in frame.columns
             else frame.l2_runs)],
        "l2_runs_names_transparent": [
            " ".join(str(int(x)) for x in r) for r in
            (frame.l2_runs_transparent if "l2_runs_transparent" in frame.columns
             else frame.l2_runs)],
        "n_switches": frame.n_switches,
        "max_run": frame.max_run,
        "total_l2_tokens": frame.total_l2_tokens,
        "l2_token_ratio": frame.l2_token_ratio.round(4),
    })
    for c in ("topic", "cs_type", "cs_function", "conversation_type"):
        if c in frame.columns:
            out[c] = frame[c].to_numpy()
    out.to_csv(path, index=False)
    return path


def save_tagged(frame: pd.DataFrame, path: str = DEFAULT_PATH) -> str:
    """Write the tagged frame to parquet, creating ``data/`` if needed."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    frame.to_parquet(path, index=False, compression="snappy")
    return path


def load_tagged(path: str = DEFAULT_PATH) -> pd.DataFrame:
    """Read back a tagged parquet, restoring ``segments`` and ``tokens_tagged``."""
    frame = pd.read_parquet(path)
    frame["segments"] = [
        [[l, int(n)] for l, n in zip(ls, ns)]
        for ls, ns in zip(frame.segment_lang, frame.segment_len)
    ]
    frame["tokens_tagged"] = [
        list(zip(ts, ls)) for ts, ls in zip(frame.tokens, frame.token_lang)
    ]
    return frame


def recompute_runs(path: str = DEFAULT_PATH) -> pd.DataFrame:
    """Recompute ``l2_runs`` from stored tokens, without re-tagging.

    Because the parquet keeps the full ``tokens``/``token_lang`` sequences, a
    change to the run-length rule does not require another pass over the
    models - the expensive part is the tagging, not the counting.
    """
    from .segments import absorb_other, l2_word_runs

    # `token_lang` is stored pre-absorption, but tag_row counts post-absorption.
    # Absorb here too, or the two paths disagree wherever a *word* carries the
    # `other` label (BERT emits it for named entities), which would split runs
    # that tag_row keeps whole.
    frame = pd.read_parquet(path)
    runs = [
        l2_word_runs(absorb_other(list(zip(ts, ls))), l2)
        for ts, ls, l2 in zip(frame.tokens, frame.token_lang, frame.second_language)
    ]
    frame["l2_runs"] = runs
    frame["n_switches"] = [len(r) for r in runs]
    frame["max_run"] = [max(r) if len(r) else 0 for r in runs]
    frame["total_l2_tokens"] = [int(sum(r)) for r in runs]
    frame.to_parquet(path, index=False, compression="snappy")
    return frame

## Load the corpus

In [ ]:
import glob
import pandas as pd

NAMES = ["Arabic_eng.csv", "Can_Eng.csv", "Chinese_eng.csv", "French_eng.csv", "German_Eng.csv",
         "German_French.csv", "Hindi_eng.csv", "Italian_eng.csv", "Japanese_eng.csv",
         "Korean_eng.csv", "Malay_eng.csv", "Philippines_eng.csv", "Russian_eng.csv",
         "Spanish_eng.csv", "Vietnamese_eng.csv"]

TOKEN = os.environ.get("HF_TOKEN")
if not TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        TOKEN = None
print("HF token:", "found" if TOKEN else "MISSING - add the HF_TOKEN secret")

def find_csvs():
    # 1) an attached Kaggle dataset
    found = {os.path.basename(p): p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True)}
    if all(n in found for n in NAMES):
        print("using attached Kaggle dataset")
        return [found[n] for n in NAMES]
    # 2) the Hugging Face Hub (gated: needs the HF_TOKEN secret)
    from huggingface_hub import snapshot_download
    snap = snapshot_download("Shelton1013/SwitchLingua_text", repo_type="dataset",
                             allow_patterns=["*.csv"], token=TOKEN)
    return [os.path.join(snap, n) for n in NAMES]

# Same order as run_tagging.load_corpus (sorted filenames), so row_id and the
# row_id % 10 split are identical to the local parquet.
files = sorted(find_csvs(), key=os.path.basename)
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(f"{len(df):,} rows")
if LIMIT:
    df = df.groupby(["first_language", "second_language"], sort=False).head(LIMIT)
    print(f"LIMIT={LIMIT}: {len(df):,} rows")
print(df.groupby(["first_language", "second_language"]).size().to_string())

## Result store (private Hugging Face dataset)

In [ ]:
# Where finished parts go. Everything already there is skipped, which is what
# makes a re-run resume.
import shutil
if UPLOAD:
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi(token=TOKEN)
    user = api.whoami()["name"]
    REPO = HF_REPO or f"{user}/switchlingua-tagged-v2"
    try:
        api.create_repo(REPO, repo_type="dataset", private=True, exist_ok=True)
    except Exception as exc:
        raise RuntimeError(f"cannot create {REPO} - does HF_TOKEN have WRITE access? {exc}")
    stored = set(api.list_repo_files(REPO, repo_type="dataset"))

    def put(local, path, msg):
        api.upload_file(path_or_fileobj=local, path_in_repo=path, repo_id=REPO,
                        repo_type="dataset", commit_message=msg)
        stored.add(path)

    def get(path):
        return hf_hub_download(REPO, path, repo_type="dataset", token=TOKEN)
    print(f"uploading to https://huggingface.co/datasets/{REPO} (private)")
else:
    REPO = os.path.join(WORK, "hf_mirror")
    os.makedirs(REPO, exist_ok=True)
    stored = {os.path.relpath(os.path.join(d, f), REPO)
              for d, _, fs in os.walk(REPO) for f in fs}

    def put(local, path, msg):
        os.makedirs(os.path.dirname(os.path.join(REPO, path)), exist_ok=True)
        shutil.copy(local, os.path.join(REPO, path))
        stored.add(path)

    def get(path):
        return os.path.join(REPO, path)
    print("UPLOAD=False: mirroring to", REPO)

CARD = '''---
license: other
pretty_name: SwitchLingua per-token language labels
---
# SwitchLingua per-token language labels

Per-token labels for [Shelton1013/SwitchLingua_text](https://huggingface.co/datasets/Shelton1013/SwitchLingua_text)
(234,172 rows). Derived from a gated dataset under its terms - keep this repo private.

* `switchlingua_tagged_rows.csv` - one line per row. `tokens` / `labels` / `entities` are
  aligned and space-joined; labels are `L1`, `L2`, `O` (punctuation/digits) - names carry
  their contextual language - and `entities` is 1 for named-entity tokens.
  `l2_runs` = length in words of each second-language run (names count as their language);
  `l2_runs_names_transparent` = the same with names transparent.
  `split` = `row_id % 10` (0-7 train, 8 val, 9 test).
* `tagged.parquet` - the same, in the local pipeline's format.
* `rows/` and `parquet/` - the per-pair parts the files above were merged from.

Taggers: Unicode script (non-Latin L1s, Devanagari Hindi), AnE-LID (Latin L1 with English
L2, romanized Hindi), wordfreq Zipf comparison (German-French), AnE-NER (entities, all rows).
'''
if "README.md" not in stored:
    card = os.path.join(WORK, "README.md")
    with open(card, "w") as fh:
        fh.write(CARD)
    put(card, "README.md", "dataset card")      # also proves the token can write
print(f"{len(stored)} files already stored")

## Load the models and plan the parts

In [ ]:
import threading, time
from src import ane_tagger
from src.persist import (ane_labels, entity_masks, export_rows_csv, row_tokens,
                         save_tagged, tag_dataframe)

ENTS = None
if ENTITY_SOURCE:
    # v1 marked every name NE and has the same tokens, so its NE positions are
    # exactly the AnE-NER masks - no need to run the NER model again.
    from huggingface_hub import hf_hub_download
    v1 = pd.read_parquet(hf_hub_download(ENTITY_SOURCE, "tagged.parquet", repo_type="dataset",
                                         token=TOKEN), columns=["row_id", "tokens", "token_lang"])
    v1 = v1.set_index("row_id")
    ENTS = {int(r): [l == "NE" for l in v1.token_lang[r]] for r in df.index}
    bad = [r for r in df.index[:: max(1, len(df) // 2000)]
           if len(row_tokens(df.data_generation_result[r])) != len(ENTS[int(r)])]
    assert not bad, f"v1 tokens differ from this tokenizer on rows {bad[:5]} - set ENTITY_SOURCE = None"
    print(f"entity masks for {len(ENTS):,} rows from {ENTITY_SOURCE} "
          f"({sum(map(sum, ENTS.values())):,} entity tokens)")
    del v1

n_gpu = torch.cuda.device_count()
DEVICES = [f"cuda:{i}" for i in range(n_gpu)] or [None]
for dev in DEVICES:                       # models on every GPU, so any part can shard
    if ENTS is None:
        ane_tagger.load(ane_tagger.NER_REPO, device=dev)
    ane_tagger.load(ane_tagger.REPO, device=dev)
print("models loaded on", DEVICES)

# Work units: one per pair, big pairs cut into PART_ROWS-row parts; smallest first,
# so finished pairs start appearing on the Hub quickly.
units = []
for (l1, l2), g in sorted(df.groupby(["first_language", "second_language"]), key=lambda x: len(x[1])):
    for k, start in enumerate(range(0, len(g), PART_ROWS)):
        units.append((f"{l1}_{l2}", k, g.iloc[start:start + PART_ROWS]))
todo = [u for u in units if f"parquet/{u[0]}/part-{u[1]:03d}.parquet" not in stored]
print(f"{len(units)} parts, {len(units) - len(todo)} already stored, {len(todo)} to run")

## Label and upload, one part at a time
About 1–2 h on T4×2 (not yet measured). A pair's files appear on the Hub as soon as it finishes.

In [ ]:
def label_part(sub):
    # Both model passes, sharded across the GPUs, then tagging.
    shards = [sub.iloc[i::len(DEVICES)] for i in range(len(DEVICES))]
    out, errors = [None] * len(DEVICES), []

    def work(i):
        try:
            dev = DEVICES[i]
            if ENTS is not None:
                e = {int(r): ENTS[int(r)] for r in shards[i].index}
            else:
                e = entity_masks(shards[i], cache=f"data/entities.{i}.parquet", device=dev,
                                 batch=BATCH, progress=False)
            a = ane_labels(shards[i], cache=f"data/ane_lid.{i}.parquet", device=dev,
                           entities=e, window=WINDOW, batch=BATCH, progress=False)
            out[i] = (e, a)
        except Exception as exc:
            errors.append(exc)

    threads = [threading.Thread(target=work, args=(i,)) for i in range(len(DEVICES))]
    [t.start() for t in threads]
    [t.join() for t in threads]
    if errors:
        raise errors[0]
    ents = {k: v for e, _ in out for k, v in e.items()}
    lid = {k: v for _, a in out for k, v in a.items()}
    return tag_dataframe(sub, progress=False, entities=ents, ane=lid)

os.makedirs("parts", exist_ok=True)
t_all, rows_done, rows_total = time.time(), 0, sum(len(u[2]) for u in todo)
for pair, k, sub in todo:
    t0 = time.time()
    tagged = label_part(sub)
    stem = f"{pair}/part-{k:03d}"
    pq_local = os.path.join("parts", stem.replace("/", "__") + ".parquet")
    csv_local = pq_local.replace(".parquet", ".csv")
    save_tagged(tagged, pq_local)
    export_rows_csv(tagged, csv_local)
    put(csv_local, f"rows/{stem}.csv", f"{pair} part {k}")
    put(pq_local, f"parquet/{stem}.parquet", f"{pair} part {k}")   # written last = part done
    rows_done += len(sub)
    rate = rows_done / (time.time() - t_all)
    print(f"{pair:24} part {k}: {len(sub):>6,} rows in {(time.time() - t0) / 60:5.1f} min | "
          f"{rows_done:,}/{rows_total:,} rows, eta {(rows_total - rows_done) / rate / 60:.0f} min",
          flush=True)
print("all parts stored")

## Merge all parts into the final CSV and parquet

In [ ]:
# Merge every stored part - including ones uploaded by an earlier, crashed run -
# into the two final files, and upload them.
parts = sorted(p for p in stored if p.startswith("parquet/") and p.endswith(".parquet"))
expected = {f"parquet/{u[0]}/part-{u[1]:03d}.parquet" for u in units}
missing = expected - set(parts)
assert not missing, f"{len(missing)} parts missing - re-run the notebook: {sorted(missing)[:5]}"

full = pd.concat([pd.read_parquet(get(p)) for p in sorted(expected)], ignore_index=True)
full = full.sort_values("row_id").reset_index(drop=True)
save_tagged(full, os.path.join(WORK, "tagged.parquet"))
csv_path = export_rows_csv(full, os.path.join(WORK, "switchlingua_tagged_rows.csv"))
print(f"merged {len(full):,} rows; CSV {os.path.getsize(csv_path) / 1e6:.0f} MB")
put(os.path.join(WORK, "tagged.parquet"), "tagged.parquet", "merged parquet")
put(csv_path, "switchlingua_tagged_rows.csv", "merged CSV - all languages")
print("uploaded the merged CSV and parquet")

## Checks and per-pair summary

In [ ]:
import numpy as np
rows = pd.read_csv(csv_path)

# tokens, labels and entity flags are aligned; labels use only L1/L2/O
bad = [r for r, t, l, e in zip(rows.row_id, rows.tokens.fillna(""), rows.labels.fillna(""),
                               rows.entities.fillna(""))
       if not len(t.split(" ")) == len(l.split(" ")) == len(e.split(" "))]
assert not bad, f"{len(bad)} rows with misaligned tokens/labels/entities, e.g. {bad[:5]}"
codes = pd.Series(" ".join(rows.labels.fillna("")).split()).value_counts()
assert set(codes.index) <= {"L1", "L2", "O"}, codes
assert len(rows) == len(df) and sorted(rows.row_id) == sorted(df.index)
print("checks passed: aligned tokens/labels/entities, 3 label codes, every row present\n")
print("label counts:", codes.to_dict(), "\n")

def runs(s):
    return [int(x) for x in str(s).split()] if isinstance(s, str) and s else []

summary = []
for (l1, l2), g in rows.groupby(["first_language", "second_language"]):
    r = [x for s in g.l2_runs for x in runs(s)]
    rt = [x for s in g.l2_runs_names_transparent for x in runs(s)]
    ra = [x for s in g.l2_runs_names_script for x in runs(s)]
    lab = " ".join(g.labels.fillna("")).split()
    ent = " ".join(g.entities.fillna("")).split()
    words = sum(1 for x in lab if x != "O")
    summary.append(dict(pair=f"{l1}->{l2}", rows=len(g),
                        name_share=f"{ent.count('1') / max(words, 1):.1%}",
                        switches=len(r), script=len(ra), transparent=len(rt),
                        median_run=float(np.median(r)) if r else 0,
                        median_transparent=float(np.median(rt)) if rt else 0,
                        single_word=f"{np.mean(np.array(r) == 1):.1%}" if r else "-"))
print(pd.DataFrame(summary).to_string(index=False))

## Examples

In [ ]:
# One example row per language pair; names are marked with *.
for (l1, l2), g in rows.groupby(["first_language", "second_language"]):
    r = g.iloc[0]
    trip = list(zip(r.tokens.split(" "), r.labels.split(" "), r.entities.split(" ")))[:30]
    print(f"{l1} -> {l2}  (row {r.row_id}, l2_runs: {r.l2_runs})")
    print("   ", " ".join(f"{t}/{c}{'*' if e == '1' else ''}" for t, c, e in trip), "\n")